# 03 · Skórování nových faktur a review queue

**Scénář ze slide 20:** _most mezi daty a byznysem_ · _AI quickstart pattern_

Tento notebook už cílí přímo na business uživatele — controller dostane
**seřazený seznam faktur ke schválení / odmítnutí** s vysvětlením, proč
je model označil. Žádné F1 score, žádné konfuzní matice.

## 3.1 Načtení modelu a nových faktur

In [ ]:
import os, sys, joblib, io
sys.path.insert(0, '../src')
from invoice_features import (load_invoices, build_features, explain_row,
                              FEATURE_COLUMNS)
import pandas as pd

# --- Načti model ---------------------------------------------------
if os.environ.get('AWS_S3_BUCKET'):
    from invoice_features import get_s3_client, get_bucket
    s3 = get_s3_client()
    body = s3.get_object(Bucket=get_bucket(),
                         Key='models/invoice_anomaly_model.joblib')['Body'].read()
    artifact = joblib.load(io.BytesIO(body))
else:
    artifact = joblib.load('../data/invoice_anomaly_model.joblib')

pipe = artifact['pipeline']
print('Model loaded.  ROC-AUC =', artifact['metrics']['roc_auc'])

In [ ]:
# --- Načti nové faktury -------------------------------------------
if os.environ.get('AWS_S3_BUCKET'):
    new_df = load_invoices('invoices_new_batch.csv', s3_client=s3,
                           bucket=get_bucket())
else:
    new_df = load_invoices('../data/invoices_new_batch.csv')

print(f'Nových faktur ke skórování: {len(new_df)}')
new_df.head()

## 3.2 Skórování

Vlastní inference je jednořádek. **Featury jdou přes stejnou funkci**,
jakou jsme použili při tréninku, s předaným category_index a
vendor_freq_table z train fáze — proto je výsledek deterministický.

In [ ]:
X_new, _ = build_features(
    new_df,
    category_index=artifact['category_index'],
    vendor_freq_table=pd.Series(artifact['vendor_freq_table']),
    category_stats=pd.DataFrame.from_dict(artifact['category_stats'], orient='index'),
)

new_df['anomaly_score'] = -pipe.decision_function(X_new)
new_df['is_flagged']    = (pipe.predict(X_new) == -1).astype(int)

print(f"Flagnuto: {new_df['is_flagged'].sum()} z {len(new_df)}")

## 3.3 Review queue pro controllera

Klíčový krok pro byznys: ke každé podezřelé faktuře přidáme **lidsky
čitelný důvod**. Bez něj je model černá skříňka a uživatel ho nepoužije.

In [ ]:
# Připravíme feature dataframe ve stejném tvaru, jaký dostává explain_row
feat_df = pd.DataFrame(X_new, columns=FEATURE_COLUMNS).astype(float)
feat_df['amount_czk'] = new_df['amount_czk'].values

queue = new_df.copy()
queue['reason'] = feat_df.apply(explain_row, axis=1)

queue = (queue
    .sort_values('anomaly_score', ascending=False)
    .loc[:, ['invoice_id', 'vendor', 'amount_czk', 'issued_on',
             'po_number', 'approver', 'anomaly_score', 'reason']]
    .reset_index(drop=True))

queue.head(15)

## 3.4 Export pro controllera

Výsledek pošleme do bucketu (a do CSV pro Excel). Controller dostane
ráno e-mailem soubor s top 25 položkami ke kontrole.

In [ ]:
import pandas as pd
from datetime import datetime

stamp = datetime.now().strftime('%Y-%m-%d')
out_file = f'../data/review_queue_{stamp}.csv'
queue.head(25).to_csv(out_file, index=False)
print('Review queue uložena do:', out_file)

if os.environ.get('AWS_S3_BUCKET'):
    from invoice_features import write_csv_s3
    write_csv_s3(queue.head(25), s3, get_bucket(),
                 f'review_queue/{stamp}.csv')
    print('A taky do S3:', f'review_queue/{stamp}.csv')

## 3.5 Registrace modelu do Model Registry

Pokud controller potvrdí, že top položky dávají smysl, model přejde do
produkce. To vyřeší Red Hat AI **Model Registry** — promo `Staging` →
`Production` je jeden klik (nebo jeden API call).

Níže si ukážeme API call. Endpoint registry je k dispozici uvnitř clusteru
na `model-registry-service.<namespace>.svc.cluster.local:8080`.

In [ ]:
# --- Registrace modelu (volitelné, vyžaduje běžící Model Registry) -
import os, json, requests

registry_url = os.environ.get('MODEL_REGISTRY_URL')
if registry_url:
    payload = {
        'name': 'invoice-anomaly-detector',
        'description': 'Isolation Forest pro detekci anomálií ve fakturách.',
        'owner': 'controlling@example.com',
    }
    try:
        r = requests.post(f'{registry_url}/api/model_registry/v1alpha3/registered_models',
                          json=payload, timeout=5)
        print('Registry response:', r.status_code)
        print(r.text[:300])
    except Exception as e:
        print('Registry call failed (OK pro lokální spuštění):', e)
else:
    print('MODEL_REGISTRY_URL nenastaveno — přeskakuji.')
    print('Na clusteru: oc get route -n <ns> | grep model-registry')

## 3.6 Shrnutí — co tím controller dostal

1. Ráno mu přistál v inboxu CSV s top 25 podezřelými fakturami.
2. U každé položky vidí **proč** byla označena (kulatá částka, nový
   dodavatel s vysokou částkou, …).
3. Pokud potvrdí, že model funguje, BI tým ho promotne do Production.
4. Když přijde NIS2 auditor, evidence je kompletní:
   `git log` (kód) → MLflow run (parametry + metriky) →
   Model Registry (kdo a kdy schválil) → review_queue/*.csv (výstupy).

Tohle je celý byznys příběh slide 20, zhmotněný v jednom workbenchi.